# Lung extraction from CT or proton MRI

Use the acquired chest CT or proton lung MRI from the ANTsX tutorial, with a
binary mask overlay. Set `modality` to select the matching acquisition; the
default CT downloads Figshare file 42934234, and proton MRI uses 42934228.
These are the tutorial acquisitions, not atlas/template substitutes.
Models and priors download on first use. 3-D inference can require substantial
memory and CPU time. A local input can override either example.

Install ANTsTorch and notebook tools from the repository root before opening this notebook:
```bash
python -m pip install -e '.[notebooks]'
python -m ipykernel install --user --name python3 --display-name "Python 3 (ANTsTorch)"
```
Select that kernel and run the cells in order. All examples default to CPU.

In [ ]:
from time import perf_counter
_example_started = perf_counter()

import torch  # Import before ANTs to avoid OpenMP conflicts on macOS.
import numpy as np
import ants
import antstorch

# Display figures inside Jupyter/VS Code even if MPLBACKEND was set to Agg.
from IPython import get_ipython
shell = get_ipython()
if shell is not None:
    shell.run_line_magic("matplotlib", "inline")
import matplotlib.pyplot as plt

device = "cpu"
antstorch.set_default_device(device)

# Synchronize the selected accelerator so timings include completed GPU work.
from contextlib import contextmanager
_example_operation_times = []

def _synchronize_example_device():
    selected = torch.device(device)
    if selected.type == "cuda":
        torch.cuda.synchronize(selected)
    elif selected.type == "mps":
        torch.mps.synchronize()

@contextmanager
def time_step(label):
    _synchronize_example_device()
    started = perf_counter()
    try:
        yield
    finally:
        _synchronize_example_device()
        seconds = perf_counter() - started
        _example_operation_times.append((label, seconds))
        print(f"{label}: {seconds:.4f} s")

In [ ]:
import os
from pathlib import Path

modality = "ct"  # "ct" or "proton"; choose the model matching the acquisition.
image_path = os.environ.get("ANTSTORCH_EXAMPLE_LUNG_IMAGE")
# Alternatively set an absolute path here. With CT, preserve Hounsfield units.
if modality not in ("ct", "proton"):
    raise ValueError("This notebook demonstrates ct and proton extraction only.")
if image_path is None:
    sample_id = "tutorialLungCT" if modality == "ct" else "tutorialLungProton"
    image_path = antstorch.get_antstorch_data(sample_id)
if not Path(image_path).is_file():
    raise FileNotFoundError(f"Input image does not exist: {image_path}")
image = ants.image_read(str(image_path)).clone("float")
assert image.dimension == 3 and not image.has_components
assert np.isfinite(image.numpy()).all() and np.ptp(image.numpy()) > 0
print("Input:", image_path, "modality:", modality, "shape:", image.shape)
with time_step('Lung extraction (result)'):
    result = antstorch.lung_extraction(image, modality=modality, device=device, verbose=True)
segmentation = result["segmentation_image"]
probabilities = result["probability_images"]
assert segmentation.shape == image.shape
assert np.isfinite(segmentation.numpy()).all()
expected_labels = [0, 1, 2, 3] if modality == "ct" else [0, 1, 2]
assert np.isin(segmentation.numpy(), expected_labels).all()
for attribute in ("spacing", "origin", "direction"):
    np.testing.assert_allclose(getattr(segmentation, attribute), getattr(image, attribute))
for probability in probabilities:
    assert probability.shape == image.shape and np.isfinite(probability.numpy()).all()

## Build a lung-only mask
Both models use 0 for background, 1 for left lung and 2 for right lung.
The CT model additionally labels airways as 3. Keep labels 1–2 for the lung
mask so that the separately labeled central airway is not included.

In [ ]:
lung_mask = ants.threshold_image(segmentation, 1, 2, 1, 0)
assert (lung_mask.numpy() > 0).any()
display_image = image.clone()
if modality == "ct":
    display_image[display_image < -1000] = -1000
    display_image[display_image > 400] = 400
# Only the display image is clipped; CT inference receives the original HU.
ants.plot(display_image, overlay=lung_mask, axis=2, nslices=8,
          overlay_cmap="autumn", overlay_alpha=0.4)
voxel_volume = float(np.prod(image.spacing))
for label, name in [(1, "Left lung"), (2, "Right lung")]:
    volume = np.count_nonzero(segmentation.numpy() == label) * voxel_volume
    print(f"{name}: {volume:.1f} cubic physical units")
output_dir = Path("example_outputs/lung_extraction") / modality
output_dir.mkdir(parents=True, exist_ok=True)
ants.image_write(segmentation, str(output_dir / "labels.nii.gz"))
ants.image_write(lung_mask, str(output_dir / "lung_mask.nii.gz"))
for label, probability in enumerate(probabilities):
    ants.image_write(probability, str(output_dir / f"probability_{label}.nii.gz"))
print("Outputs (matching files are overwritten):", output_dir.resolve())

Inspect left/right orientation, pleural boundaries, pathology and coverage.
CT and proton MRI use different models and preprocessing. Do not feed a CT
to the proton model or normalize CT values before the CT workflow.
Other `lung_extraction` branches include lobes, ventilation and X-ray;
their input dimensions and return structures differ from this example.

## Execution time

Run **Restart Kernel and Run All Cells** for a comparable end-to-end duration.
The total starts at the first setup cell and includes imports, downloads, data
preparation, computation, plotting and file writes. Kernel startup is excluded;
manual pauses between cells are included. Operation timings include any
preprocessing or asset downloads performed inside the called function. First
runs can be slower because caches are empty. Timings describe this run on the
selected device, not a fixed performance estimate for other machines.

In [ ]:
_synchronize_example_device()
total_seconds = perf_counter() - _example_started
operation_seconds = sum(seconds for _, seconds in _example_operation_times)
print(f"Total elapsed time: {total_seconds:.2f} s ({total_seconds / 60:.2f} min)")
print(f"Timed operations: {operation_seconds:.4f} s on {device}")